Check how the autoencoder outputs look across runs and seeds.  
The goal is to find high impact stable components.  Generally, another simple way is to simply take the minimum loss components from your elbow.  

But still, this notebook will show the components.  


I seem to have a reasonable way now. Im basically only doing the minimum losses.  
I'm not sure if i should cluster the components, lets see. Stability score might be useful if there are more than one score, basically clustering is useful if we have more than one score. would be a good metric in summary. We only do for the low loss ones.   

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np

def sample_multipliers(
    n_samples,
    eps_multiplier_range=(0.1, 10.0),
    alpha_range=(5_000,10_000),
    ratio_range=(0.1, 10.0),
    rng_seed=42,
):
    rng = np.random.default_rng(rng_seed)
    eps_multipliers = rng.uniform(*eps_multiplier_range, size=n_samples)
    ratios = rng.uniform(*ratio_range, size=n_samples)
    alpha_ranges = rng.uniform(*alpha_range, size=n_samples)
    return list(zip(eps_multipliers, ratios, alpha_ranges))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.mnist import SimpleMNIST
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
# from pt_to_api.benchmark.thresholding import threshold_assuming_noise_at_0_with_only_one_side_active
from pt_to_api.benchmark import thresholding as TR
import gc
import pickle
from pt_to_api.utils import otsu_threshold
from sklearn.preprocessing import normalize
from collections import defaultdict
import math
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import json
from dataclasses import dataclass
from pt_to_api import benchmark as B
from sklearn.mixture import GaussianMixture

In [ ]:
# MODEL_PATH = Path("../../../pt-to-api/data/model.pt")
# INPUT_PATH = Path("../../../pt-to-api/data/first-input-tens.pt")

# inp = torch.load(INPUT_PATH, weights_only=False)
# model = SimpleMNIST()
# model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
MODE = "light"
if MODE == "dark":
    plt.style.use('dark_background')
else:
    plt.style.use('default')


In [ ]:
DRIVE_PATH = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist")
MAIN_OUT_DIR = (DRIVE_PATH / "collect-patches" / "data")

layer_name = "layers.2"
channel = 5

In [ ]:
from pt_to_api.benchmark.utils import support_overlap_matrix_batched

def get_support_overlap_of_run(run):
    # run.codes: [samples, n-comps]
    # components: [n-comps, dims]
    # need: [s,c,d]
    latents = np.einsum("sc,cd->scd", run.codes, run.components)
    overlap_matrix = supaport_overlap_matrix_batched(latents, threshold=0.1)
    overlap_matrix = overlap_matrix.fill_diagonal_(0)

    # this is the mean support overlap

    return overlap_matrix

In [ ]:
from pt_to_api.benchmark.core import SingleRun, RunId, CompId, IdAndComp, C2R, LazySingleRun
from pt_to_api.benchmark.jax import Autoencoder
from pt_to_api.benchmark.scalers import NormaliseStdScaler
from sklearn.mixture import GaussianMixture
from kneed import KneeLocator
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm





def _mse(x, codes, comps):
    diff = x - (codes@comps)
    return (diff**2).mean()


def get_comp_scores(X, codes, components):
    main_mse = _mse(X, codes, components)
    scores = []
    for i in range(len(components)):
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(X, codes, comps)
        comp_score = new_mse - main_mse
        scores.append(comp_score)
    return scores, main_mse


def get_comp_scores_when_active(X, codes, components, verbose=False):
    scores, active_ratios = [], []
    for i in range(len(components)):
        try:
            idxs = get_indices_where_comp_is_active(i, codes, components)
        except TR.ThresholdFailureException as ex:
            # now what?
            # we dont have indices for ith component
            # we would like to assign a score
            # in this case, we assign 0
            # we cant find the comp only so its fine
            if verbose:
                print(f"WARN: could not find threshold for comp={i} total_components={len(components)}")
                print(f"\tcause: {ex}")
                print(f"\tsign result: {ex.sign_result}")
            scores.append(0)
            active_ratios.append(0)
            continue

        _X, _codes = X[idxs], codes[idxs]

        main_mse = _mse(_X, _codes, components)
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(_X, _codes, comps)
        comp_score = (new_mse - main_mse) / new_mse

        scores.append(comp_score)
        active_ratios.append(len(idxs) / codes.shape[0])
    return scores, active_ratios

def find_threshold_using_gmm(values, gap_ratio_to_add=0):
    # assume values are always positive
    if np.any(values < 0):
        raise Exception("values for finding threshold should always be positive")
    values = np.abs(np.array(values))
    gmm = GaussianMixture(n_components=2, random_state=0)
    gmm.fit(values.reshape(-1, 1))
    
    garbage_idx = np.argmin(gmm.means_)  # garbage is near 0, so smallest mean
    real_idx = 1 - garbage_idx
    
    midpoint = (gmm.means_[garbage_idx] + gmm.means_[real_idx]) / 2
    gap = gmm.means_[real_idx] - gmm.means_[garbage_idx]
    return (midpoint + gap_ratio_to_add * gap).item()

def get_indices_where_comp_is_active(comp_idx, codes, components):
    # it might be useful to actually persist these calculated thresholds
    comp = components[comp_idx]
    data = codes[:, comp_idx]

    thresh_state = TR.threshold_assuming_noise_at_0_with_only_one_side_active(data)
    thresh = None
    match thresh_state:
        case TR.Ambiguous() | TR.MixedSign():
            raise TR.ThresholdFailureException(f"failure in finding threshold comp_idx={comp_idx}", thresh_state)
        case TR.NonNegative(threshold = t): 
            thresh = t
        case TR.NonPositive(threshold = t):
            thresh = t
    if thresh is None:
        raise Exception("thresh cant be none, the match statement is not exhaustive")

    indices = np.argwhere(np.abs(data) >= np.abs(thresh))
    return indices

def get_weights_and_patches(layer_data_dir):
    weight = torch.load(layer_data_dir / "weight.pt", weights_only=False)
    patches = torch.load(layer_data_dir / "samples.pt", weights_only=False)
    return weight, patches

def get_loaded_normaliser(file_path):
    normaliser = NormaliseStdScaler()
    state = load_normaliser_state(file_path)
    normaliser.global_std_ = state["global_std_"]
    return normaliser

def load_normaliser_state(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

def load_samples_used_for_training(layer_data_dir):
    weight, patches = get_weights_and_patches(layer_data_dir)
    scaler = get_loaded_normaliser(layer_data_dir / "normaliser.json")
    pw = weight * patches
    scaled_pw = scaler.transform(pw)
    return scaled_pw, scaler


def load_c2r(runs_dir: Path) -> C2R:
    c2r = {}
    for p in runs_dir.rglob("*.pt"):
        # print("path", p)
        seed = int(p.stem.split("_")[1])
        n_components = int(p.parent.stem)
        c2r[RunId(n_components, seed)] = LazySingleRun(p)
        # c2r[RunId(n_components, seed)] = torch.load(p, weights_only=False)
    return c2r

def filter_c2r_only_comps_allowed(c2r:C2R, comps: list[int]) -> C2R:
    return {
        run_id: run for run_id, run in c2r.items() if run_id.n_components in comps
    }

def get_serialised_components_and_their_lookup_table(c2r: C2R) -> tuple[list[np.ndarray], dict[int, CompId]]:
    idx_by_comp_id = {}
    all_comps = []
    for run_id, run in c2r.items():
        for comp_idx, comp in enumerate(run.components):
            idx = len(all_comps)
            all_comps.append(comp)
            idx_by_comp_id[idx] = CompId(run_id, comp_idx)
    return all_comps, idx_by_comp_id

def get_distance_matrix(all_comps: list[np.ndarray]) -> np.ndarray:
    X_reduced = normalize(all_comps, "l2")
    abs_sim = np.abs(cosine_similarity(X_reduced, X_reduced))  # (n_samples, n_samples)
    distance_matrix = 1 - abs_sim  # values in [0, 1]
    return distance_matrix


def get_comp2score(c2r: C2R, scaled_pw: np.ndarray) -> dict[CompId, float]:
    comp2score_when_active = {}
    for run_id, run in tqdm(c2r.items()):
        scores, active_ratios = get_comp_scores_when_active(scaled_pw, run.codes, run.components)
        scores = np.array(scores)
        for j in range(len(scores)):
            comp2score_when_active[CompId(run_id, j)] = (scores[j], active_ratios[j])
    return comp2score_when_active


def get_label_by_comps(
    cluster_labels: list[int], all_comps: list[np.ndarray], idx_by_comp_id: dict[int, CompId]
) -> dict[str, list[IdAndComp]]:
    l2comps = defaultdict(list)
    for i, label in enumerate(cluster_labels):
        comp_id = idx_by_comp_id[i]
        l2comps[label].append(IdAndComp(comp_id, all_comps[i]))
    return l2comps


def get_label_by_highest_scored_comp(
    label_by_comps, comp2score
):
    lable2comp = {}
    for label, id_and_comp_list in label_by_comps.items():
        scores = [
            comp2score[comp_id]
            for comp_id, _ in id_and_comp_list
        ]
        max_score_idx = np.argmax(scores)
        max_score = scores[max_score_idx]
        lable2comp[label] = (id_and_comp_list[max_score_idx], max_score)
    return label2comp

def show_label_by_comps(label_by_comps, comp2score, comp_shape):
    for label, id_and_comp_list in label_by_comps.items():
        # print("hahahaha", id_and_comp_list)
        comps, titles, scores = [], [], []
        for id_and_comp in id_and_comp_list:
            comp_id, comp = id_and_comp.comp_id, id_and_comp.comp
            comps.append(comp.reshape(comp_shape))
            score, active_ratio = comp2score[comp_id]
            titles.append(f"{comp_id.run_id.n_components} ({score:.3f}/{active_ratio:.3f})")
            scores.append(score)
        p50_score = np.median(scores)
        p75_score = np.percentile(scores, 75)
        cols = min(len(comps), 10)
        rows = math.ceil(len(comps) / cols)
        S(comps, (20,rows*3), cols, suptitle=f"{label} p50: {p50_score:.4f} p75: {p75_score:.4f}", ax_titles=titles)
        plt.show()


def get_n2seeds(c2r: C2R) -> dict[int, list[int]]:
    n2seeds = defaultdict(list)
    for c in c2r:
        n2seeds[c.n_components].append(c.seed)
    n2seeds = {n: list(sorted(seeds)) for n, seeds in n2seeds.items()}
    return n2seeds


def get_n2runs(c2r: C2R) -> dict[int, list]:
    n2seeds = get_n2seeds(c2r)
    def _comp_list(n, seeds):
        return [c2r[RunId(n, seed)] for seed in seeds]
    return {n: _comp_list(n, seeds)  for n, seeds in n2seeds.items()}

def plot_losses(c2r: C2R) -> None:
    comps_list = sorted(set([c.n_components for c in c2r]))
    print("components", comps_list)
    n2seeds = get_n2seeds(c2r)
    losses = []
    for n in comps_list:
        min_loss_of_comp = np.min([c2r[RunId(n, seed)].loss for seed in n2seeds[n]])
        losses.append(min_loss_of_comp)
    plt.plot(comps_list, losses)
    plt.show()

    # min loss seed index
    min_loss_idx = np.argsort(losses)[0]
    return comps_list[min_loss_idx]

def show_single_run(run, comp2score, n, seed, image_shape):
    cols = min(len(run.components), 6)
    row_sz = math.ceil(len(run.components) / cols)*3
    ax_titles = []
    scores = []
    if comp2score is not None:
        for i in range(len(run.components)):
            score, active_ratio = comp2score[CompId(RunId(n, seed), i)]  
            scores.append(score)
            ax_titles.append(f"{score:.3f}/{active_ratio:.3f}")
        idxs = list(reversed(np.argsort(scores)))
    else:
        idxs = list(range(len(run.components)))
    
    S(
        [run.components[i].reshape(image_shape) for i in idxs],
        (20, row_sz),
        cols,
        ax_titles=[ax_titles[i] for i in idxs],
        suptitle=f"{seed}: {run.loss}",
        mode=MODE,
        viztype="local"
    )
    plt.show()


def fit_gmm_auto(losses, k_range=(2, 10), plot=True):
    
    losses = np.array(losses)
    if len(losses.shape) == 1:
        losses = losses.reshape(-1, 1)
    if k_range[1] > losses.shape[0]:
        k_range = k_range[0], losses.shape[0]
    ks = range(*k_range)
    bics = []
    for k in ks:
        gmm = GaussianMixture(n_components=k, random_state=42)
        gmm.fit(losses)
        bics.append(gmm.bic(losses))

    best_k = ks[np.argmin(bics)]


    gmm = GaussianMixture(n_components=best_k, random_state=42)
    gmm.fit(losses)
    labels = gmm.predict(losses)
    probs = gmm.predict_proba(losses)

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        
        axes[0].plot(ks, bics, marker='o')
        axes[0].axvline(best_k, color='red', linestyle='--', label=f'elbow k={best_k}')
        axes[0].set_xlabel('mixture-count')
        axes[0].set_ylabel('BIC')
        axes[0].set_title('BIC vs k')
        axes[0].legend()

        # Loss scatter colored by cluster
        scatter = axes[1].scatter(range(len(losses)), losses, c=labels, cmap='tab10', zorder=5)
        plt.colorbar(scatter, ax=axes[1], label='Cluster')
        for i, (loss, label) in enumerate(zip(losses.flatten(), labels)):
            axes[1].annotate(str(label), (i, loss), textcoords="offset points", xytext=(0, 5), fontsize=7)
        axes[1].set_xlabel('Sample index')
        axes[1].set_ylabel('Loss')
        axes[1].set_title(f'GMM clusters (k={best_k})')

        plt.tight_layout()
        plt.show()

    return gmm, labels, probs, best_k

In [ ]:
# one last thing
# i would like to know how many comp2scores have dead atoms
# and raise an exception if the min loss one has a dead atom
from pt_to_api.benchmark.utils import hungarian_match



def get_run_ids_with_any_0_score_atom(c2r, c2score):
    res = set()
    for run_id, run in c2r.items():
        found = False
        for comp_idx in range(len(run.components)):
            comp_id = CompId(run_id, comp_idx)
            score = c2score[comp_id]
            if score[0] == 0 and score[1] == 0:
                found = True
                break
        if found:
            res.add(run_id)
    return res


def show_analysis_for_comp(layer_data_dir, c2r, comp, image_shape, max_samples_to_show=1, fc2score=None, loss_threshold=None):
    fc2r = filter_c2r_only_comps_allowed(c2r, [comp])
    fn2seeds = get_n2seeds(fc2r)
    if fc2score is None:
        scaled_samples, scaler = load_samples_used_for_training(layer_data_dir)
        fc2score = get_comp2score(fc2r, scaled_samples)

    if loss_threshold is None:
        res_seeds, res_runs, res_losses = get_min_loss_runs_using_gmm(fc2r, fn2seeds, comp)
    else:
        res_seeds, res_runs, res_losses = get_runs_below_loss(fc2r, fn2seeds, comp, loss_threshold)
    if len(res_runs) > 1:
        _, stability_score, most_similar_idx, _ = hungarian_match([r.components for r in res_runs])
    else:
        print("WARN: number of runs with minimal loss = 1, cant do similarity matching")
        stability_score = -1
        most_similar_idx = 0
    run_ids_with_dead_atoms = get_run_ids_with_any_0_score_atom(fc2r, fc2score) 
    print("############################# SUMMARY #####################################")
    print("n_components", comp)
    print("minimum loss", res_losses[0])
    print("support overlap on min loss run", get_support_overlap_of_run(res_runs[0]))
    print("overall stability score", stability_score)
    print("minimum loss seed", res_seeds[0])
    print("most similar seed", res_seeds[most_similar_idx])
    print("support overlap on most similar seed", get_support_overlap_of_run(res_runs[most_similar_idx]))
    
    print("number of run ids with at least 1 0-score atoms", len(run_ids_with_dead_atoms), "total =", len(list(fc2r.values())))

    print("############################ least loss run ###############################")
    show_single_run(res_runs[0], fc2score, comp, res_seeds[0], image_shape)
    print("############################# most stable run ##############################")
    show_single_run(res_runs[most_similar_idx], fc2score, comp, res_seeds[most_similar_idx], image_shape)

    print("########################### min loss run inputs and details ################")
    show_recons_and_data(layer_data_dir, res_runs[0], (3,3))
    
    print("############################# runs other than stable and top ###############")
    to_show_res_seeds = get_first_n_samples(res_seeds, max_samples_to_show, [0, most_similar_idx])
    to_show_res_runs = get_first_n_samples(res_runs, max_samples_to_show, [0, most_similar_idx])
    to_show_res_losses = get_first_n_samples(res_losses, max_samples_to_show, [0, most_similar_idx])
    _show_runs(to_show_res_seeds, to_show_res_runs, comp, fc2score, image_shape)


    
    min_loss_run_id = RunId(comp, res_seeds[0])
    if min_loss_run_id in run_ids_with_dead_atoms:
        print(
            f"the minimum loss run-id={min_loss_run_id} has a at least one atom with 0-score, probably a dead atom\n",
            "you probably want to decrease the number of components for now\n",
            "if you believe that  this might be the ideal components, this might an issue with training\n",
        )
    return fc2r, fc2score


def get_runs_below_loss(fc2r, fn2seeds, comp, loss_threshold):
    seeds = sorted(fn2seeds[comp])
    runs = [fc2r[RunId(comp, seed)] for seed in seeds]
    losses = [r.loss for r in runs]
    idxs = np.argsort(losses)
    res_seeds, res_runs, res_losses = [], [], []
    for i in idxs:
        if losses[i] > loss_threshold:
            break
        res_seeds.append(seeds[i])
        res_losses.append(losses[i])
        res_runs.append(runs[i])
    return res_seeds, res_runs, res_losses

def get_first_n_samples(samples, n=-1, exclude=None):
    exclude = set(exclude or [])
    filtered = [s for i, s in enumerate(samples) if i not in exclude]
    return filtered[:n] if n != -1 else filtered

def get_min_loss_runs_using_gmm(fc2r, fn2seeds, comp, plot=True):
    seeds, runs, labels, losses = _show_gmm_plot_and_get_labels(fc2r, fn2seeds, comp, plot=plot)
    min_loss_label = _label_with_min_loss(labels, losses)
    idxs = np.argwhere(labels == min_loss_label)
    idxs = [i.item() for i in idxs]

    res_seeds = [seeds[i] for i in idxs] 
    res_losses = [losses[i] for i in idxs]
    res_runs = [runs[i] for i in idxs]
    res_seeds, res_runs, res_losses = _sorted_by_loss(res_seeds, res_runs, res_losses)
    return res_seeds, res_runs, res_losses

def _sorted_by_loss(seeds, runs, losses):
    idxs = np.argsort(losses)
    res_seeds, res_runs, res_losses = [], [], []
    for i in idxs:
        res_seeds.append(seeds[i])
        res_runs.append(runs[i])
        res_losses.append(losses[i])
    return res_seeds, res_runs, res_losses

def _show_runs(seeds, runs, comp, c2score, image_shape):
    for seed, run in zip(seeds, runs):
        show_single_run(run, c2score, comp, seed, image_shape)

def _label_with_min_loss(labels, losses):
    label2losses = defaultdict(list)
    for label, loss in zip(labels, losses):
        label2losses[label].append(loss)

    label2minloss = {label: np.min(losses) for label, losses in label2losses.items()}
    min_val = float("inf")
    min_label = None
    for label, loss in label2minloss.items():
        if loss < min_val:
            min_val = loss
            min_label = label
    return min_label


def _show_gmm_plot_and_get_labels(fc2r, fn2seeds, comp, plot=True):
    seeds = fn2seeds[comp]
    runs = [fc2r[RunId(comp, seed)] for seed in seeds]
    losses = [r.loss for r in runs]

    _, labels, _, _ = fit_gmm_auto(losses, plot=plot)
    return seeds, runs, labels, losses

def get_run_ids_with_at_least_0_score(c2score):
    run_ids = set()
    for comp_id, score in c2score.items():
        if score[0] == 0 and score[1] == 0:
            run_ids.add(comp_id.run_id)
    return run_ids

In [ ]:

def _plot_stability_scores(c2r):
    comps_list = sorted(set([c.n_components for c in c2r]))
    scores = []
    xs = []
    for comp in tqdm(comps_list):
        fc2r = filter_c2r_only_comps_allowed(c2r, [comp])
        fn2seeds = get_n2seeds(fc2r)
        res_seeds, res_runs, res_losses = get_min_loss_runs_using_gmm(fc2r, fn2seeds, comp, plot=False)
        if len(res_runs) < 2:
            print(f"WARN: got less than 2 runs for stability calculation, skipping: comp={comp}")
            continue
        _, stability_score, most_similar_idx, _ = hungarian_match([r.components for r in res_runs])
        xs.append(comp)
        scores.append(stability_score)
    plt.plot(xs, scores)
    plt.title("stability scores")
    plt.show()

def auto_basic_analyse(main_out_dir, layer_name, channel, shape, max_samples_to_show=0, loss_threshold=None, force_comp=None, runs_key="runs"):
    gc.collect()
    
    
    layer_data_dir = main_out_dir / layer_name / str(channel)
    runs_dir =  layer_data_dir / runs_key
    
    print("loading runs")
    c2r = load_c2r(runs_dir)
    min_loss_comp = plot_losses(c2r)
    _plot_stability_scores(c2r)
    if force_comp is None:
        print("using minimum loss: component =", min_loss_comp)
    else:
        print("using user passed component =", force_comp)
        min_loss_comp = force_comp

    show_analysis_for_comp(layer_data_dir, c2r, min_loss_comp, shape, max_samples_to_show=max_samples_to_show, loss_threshold=loss_threshold)

In [ ]:
# check some reconstructions
import math

def _calc_figsize(total_comps, comp_col_size=3, comp_row_size=3):
    comp_col_size = 3
    comp_row_size = 3
    # ncols = min(len(components), 8)
    ncols = min(total_comps, 8)
    nrows = math.ceil(total_comps / ncols)
    figsize = (ncols*comp_col_size, nrows*comp_row_size)
    return figsize, ncols

def show_recons_and_data(scaled_samples_or_layer_data_dir, run, image_shape, samples_to_show=5, inputs_to_show=10):
    if isinstance(scaled_samples_or_layer_data_dir, Path):
        scaled_samples, _ = load_samples_used_for_training(scaled_samples_or_layer_data_dir)
    else:
        scaled_samples = scaled_samples_or_layer_data_dir
    recon = run.recon
    codes = run.codes
    components = run.components

    w_comps_figsize, w_comps_ncols = _calc_figsize(len(components))
    samples_figsize, samples_ncols = _calc_figsize(inputs_to_show)

    S([c.reshape(image_shape) for c in components], w_comps_figsize, w_comps_ncols, suptitle="raw components")
    plt.plot()


    idxs = np.random.permutation(len(scaled_samples))[:inputs_to_show]
    S([scaled_samples[i].reshape(image_shape) for i in idxs], samples_figsize, samples_ncols, suptitle="samples")

    idxs = np.random.permutation(len(recon))[:samples_to_show]
    for i in idxs:
        sam, rec, code = scaled_samples[i], recon[i], codes[i]
        comps = [code[j] * components[j] for j in range(code.shape[0])]
        print(f"######################### {i} ###############################")
        S([sam.reshape(image_shape), rec.reshape(image_shape)], ax_titles=["original", "recons"])
        plt.show()
        S([code.reshape(1,-1)], (7, 3), suptitle="codes")
        plt.show()
        S([c.reshape(image_shape) for c in comps], w_comps_figsize, w_comps_ncols, suptitle="weighted components")
        plt.show()

# layers.2



In [ ]:
L2_NAME = "layers.2"
L2_SHAPE = (8,9)

## 0 ✅

- n-comps: 14
- seeds: 10


In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 0, L2_SHAPE)

## 1 ✅

- n-comps: 10
- seeds: 25

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 1, L2_SHAPE)

## 2 ✅

- n-comps: 11
- seed: 24

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 2, L2_SHAPE, force_comp=11)

## 3 ✅

- n-comps: 10
- seeds: 6

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 3, L2_SHAPE)

## 4 ✅

- n-components: 9
- seed: 35

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 4, L2_SHAPE)

## 5 ✅

- n-components: 9
- seeds: 38

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 5, L2_SHAPE, loss_threshold=0.2)

## 6 ✅

- comps=12
- seeds: 38

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 6, L2_SHAPE, loss_threshold=0.225)

## 7 ✅

- comps: 8
- seed: 25

loss is quite high actually, 0.2 is not nice. it is also decreasing later, will need to check that out.  
Stability is fine.  

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 7, L2_SHAPE)

## 8 ✅ (very good component)

- comps: 11
- seed: 36

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 8, L2_SHAPE)

## 9 ✅
- comps: 12
- seed: 2

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 9, L2_SHAPE)

## 10 ✅
- comps: 11
- seed: 46

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 10, L2_SHAPE)

## 11 ✅

- comps: 10
- seed: 36

11 might need more training? this is quite bad.  
It might have a shit load of components it seems

Well this one sucks. Will do 10 for now

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 11, L2_SHAPE, force_comp=10)

## 12 ✅

- comps: 11
- seed: 8 (using the most stable one here)


In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 12, L2_SHAPE, force_comp=11)

## 13 ✅

- n-comps: 14
- seed: 33

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 13, L2_SHAPE)

## 14 ✅

- comps: 12
- seed: 47

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 14, L2_SHAPE, loss_threshold=0.225)

## 15 ✅

- comps: 14
- seed: 27

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, L2_NAME, 15, L2_SHAPE)

# layers.0

The stuff below is garbage. the components had a lot of overlap ;_;

## 0 ✅


- n-comps = 2
- seed = 1


We are picking the elbow now it sees. The loss is extremely low, going to `7` is not that needed it seems.  Although, i might miss some values. Lets compare them

In [ ]:
# from pt_to_api.benchmark.core import LazySingleRun
layer_data_dir = MAIN_OUT_DIR / "layers.0" / "0"
show_recons_and_data(layer_data_dir, LazySingleRun(layer_data_dir / "runs" / "5" / "seed_.pt"), (3,3), inputs_to_show=16)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 0, (3,3), force_comp=5)

## 1 ✅

- n-components=3
- seed=1

In [ ]:
# from pt_to_api.benchmark.core import LazySingleRun
layer_data_dir = MAIN_OUT_DIR / "layers.0" / "1"
show_recons_and_data(layer_data_dir, LazySingleRun(layer_data_dir / "runs" / "3" / "seed_1.pt"), (3,3), inputs_to_show=16)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 1, (3,3), force_comp=3)

## 2 ✅

- n-components=2
- seed=29

In [ ]:
# from pt_to_api.benchmark.core import LazySingleRun
layer_data_dir = MAIN_OUT_DIR / "layers.0" / "2"
show_recons_and_data(layer_data_dir, LazySingleRun(layer_data_dir / "runs" / "2" / "seed_29.pt"), (3,3), inputs_to_show=16)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 2, (3,3), force_comp=2)

## 3 ✅

- n-components=4
- seed=41

In [ ]:
import pt_to_api.benchmark.jax as JX
from pt_to_api.benchmark.init_strats import NoInitStrategy
from pt_to_api.benchmark.anneal import CosineAnnealReconError

In [ ]:
layer_data_dir = MAIN_OUT_DIR / "layers.0" / "3"
scaled_samples, scaler = load_samples_used_for_training(layer_data_dir)

scaled_samples = scaled_samples[:1000]

In [ ]:
runs = JX.train(
    scaled_samples,
    4,
    1,
    1e-2,
    epochs=1000,
    baseline_epochs=600,
    batch_size=32,
    tensorboard_log_dir=Path("/tmp/tensorboard"),
    recon_err_schedule=CosineAnnealReconError(1300, min_factor=1, hold_frac=0),
    init_strategy=NoInitStrategy(),
)
print("loss", runs[0].loss)

In [ ]:
# we will divide each component by this, so the code needs to multiply

np.abs(runs[0].components).max(axis=1)

In [ ]:
from copy import deepcopy

run= deepcopy(runs[0])

In [ ]:
run.codes.shape, sc.shape, run.components.shape

In [ ]:
run= deepcopy(runs[0])
sc = np.abs(run.components).max(axis=1)
run.components = (run.components.T / sc).T
run.codes *= sc

In [ ]:
show_recons_and_data(scaled_samples, run, (3,3), inputs_to_show=16)

In [ ]:
from sklearn.cluster import KMeans


cluster = KMeans(n_clusters=8).fit(run.codes)

In [ ]:
S([run.recon[3].reshape(3,3), run.recon[15].reshape(3,3)])

In [ ]:
cluster.labels_

In [ ]:
# from pt_to_api.benchmark.core import LazySingleRun
layer_data_dir = MAIN_OUT_DIR / "layers.0" / "3"
show_recons_and_data(layer_data_dir, LazySingleRun(layer_data_dir / "runs" / "4" / "seed_41.pt"), (3,3), inputs_to_show=16)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 3, (3,3), force_comp=3)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 3, (3,3), force_comp=4)

## 4 ✅

- n-components=4
- seed=15

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 4, (3,3), force_comp=7)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 4, (3,3), force_comp=4)

## 5 ✅

- n-components=7
- seed=3

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 5, (3,3))

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 5, (3,3), force_comp=7)

## 6 ✅

- n-components=2
- seed=45

This kernel has problems thresholding. it has the same pattern coming up in both pos and neg values (which is important for its signals).  
So the algorithm is getting confused. will need to check that out later. for now using 2 states.  

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 6, (3,3))

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 6, (3,3), force_comp=5)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 6, (3,3), force_comp=3)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 6, (3,3), force_comp=2)

## 7 ✅

- n-components=5
- seed=18

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 7, (3,3))

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 7, (3,3), force_comp=5)

# layers.0 only pos codes

## 0

- comps: 2
- seed: 2

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 0, (3,3), runs_key="pos-only-runs", force_comp=4)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 0, (3,3), runs_key="pos-only-runs", force_comp=4)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 0, (3,3), runs_key="pos-only-runs", force_comp=2)

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 0, (3,3), runs_key="pos-only-runs", force_comp=3)

## 1

I need the overlap score also now i think.  

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 1, (3,3), runs_key="pos-only-runs", force_comp=5)

In [ ]:
# now i need the latents for this run

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 1, (3,3), runs_key="pos-only-runs", force_comp=4, loss_threshold=0.11)

In [ ]:
# from pt_to_api.benchmark.core import LazySingleRun
layer_data_dir = MAIN_OUT_DIR / "layers.0" / "0"
show_recons_and_data(layer_data_dir, LazySingleRun(layer_data_dir / "pos-only-runs" / "2" / "seed_1.pt"), (3,3), inputs_to_show=16)

### Visualise

We dont do > 15 components, they have many dead atoms. The loss curve is also quite bad.  
This component has quite unstable losses compared to K12.  

n_components = 14 looks fine.  

It is mainly a symptom of them being dead atoms.  
I might need better handling for checking dead atoms. The code currently is slightly hand wavy. But its okay.   

## 5

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
layer_name, channel = "layers.0", 5
layer_data_dir = MAIN_OUT_DIR / layer_name / str(channel)
runs_dir =  layer_data_dir / "pos-only-runs"

scaled_samples, scaler = load_samples_used_for_training(layer_data_dir)

# print("loading runs")
# c2r = load_c2r(runs_dir)
# PCA(

In [ ]:
pca = PCA(9).fit(scaled_samples[:1000])

In [ ]:
pca

In [ ]:
S([c.reshape(3,3) for c in pca.components_], 20, 9, ax_titles=[f"{a:.3f}" for a in pca.explained_variance_ratio_])
plt.show()

In [ ]:
auto_basic_analyse(MAIN_OUT_DIR, "layers.0", 5, (3,3), runs_key="pos-only-runs")

# Cluster the components

If you cluster components across all `n_components`, it can get confusing quite fast.  
It is better to find the range which is useful for analysis. And only cluster there.  


So we do a range. And its important to also remove the dead atoms. For now, we use score == 0, although this is brittle.  
But then, it also can make sense, a dead atom has zero score :)   


In [ ]:
fcomps = [runs[i].components for i, label in enumerate(labels) if label == 1]
from pt_to_api.benchmark.utils import hungarian_match

upper, stability_score, best_run_idx, pairwise_sims = hungarian_match(fcomps)
stability_score

In [ ]:
# we want the runs for these
for i, label in enumerate(labels):
    if label != 1:
        continue
    run = runs[i]
    S([c.reshape(SHAPE) for c in run.components], (20,6),  ncols=10, mode=MODE)
    plt.show()

In [ ]:
bics = []
ks = range(2, 10)
for k in ks:
    gmm = GaussianMixture(n_components=k)
    gmm.fit(losses)
    bics.append(gmm.bic(losses))

plt.plot(ks, bics)  # elbow = good k

In [ ]:
labels

In [ ]:
filtered_c2r = {}
allowed_n_comps = [9, 10]
for run_id, run in c2r.items():
    if run_id.n_components not in allowed_n_comps:
        continue
    filtered_c2r[run_id] = run
    

    ######### we dont remove dead atoms for now, they go to -1 anyways
    # for i, comp in enumerate(run.components):
    #     comp_id = CompId(run_id, i)
    #     score, _ = comp2score[comp_id]
    #     if score == 0:
    #         # dead atom
    #         continue

In [ ]:
# all_comps, idx_by_comp_id = get_serialised_components_and_their_lookup_table(filtered_c2r)

In [ ]:
all_comps, idx_by_comp_id = get_serialised_components_and_their_lookup_table(filtered_c2r)
distance_matrix = get_distance_matrix(all_comps)
hdbscan = HDBSCAN(copy=True, min_cluster_size=5, metric="precomputed")
hdbscan.fit(distance_matrix)

In [ ]:
scaled_samples, scaler = load_samples_used_for_training(layer_data_dir)

In [ ]:
label_by_comps = get_label_by_comps(hdbscan.labels_, all_comps, idx_by_comp_id)
comp2score = get_comp2score(filtered_c2r, scaled_samples)

In [ ]:
label_by_comps

In [ ]:
show_label_by_comps(label_by_comps, comp2score, (8,9))

We are in the territory of having a lot of components which are hard to look at manually. and verify what works :).   
Well kodewa kodewa this sucks.  
It might be best to train the model with comps initialised from each cluster and see which ones remain alive.  
Easy, peasy.  

Some components are not stable, and they are not high contributing.  
Lets see if we can think of a simple metric.


- First: if the `score` itself, which is now quite representative (the amount of MSE difference when the component was active) is a good indicator.  
  - We need to find the ones which score very low.  
  - Note that score finding itself is slightly mathy (it has a formula). I dont know if I can naively divide something from it.  
  - Looking at the seeds, it is clear that training did not go very well for some of them. We NEED fast seed evaluation now. I can run mass seed evals, and only use the ones which are near the minima, whatever near means.  
  - How do we do that though? I might use something like a GMM again for this (GMM thresholding FTW, the num components can be 1 or 2 now though, will need to check).  
    - For now, manual, but I'll need to test approaches.
    - The first thing is automating seed runs though, its too expensive to do multiple seeds right now.  
- Anyways, this is for reducing the components, making each run more stable. We still have the problem of what to do right now.  
  - components can have overlaps. We want to find the components which "remain" in the model after training. others would drift.  
  - now which conmponent to pick from the cluster. 
    - highest score?
    - highest similarity? -> this makes most sense
    - cluster centroid? -> might not be the best bet, im not sure.  
  - similarity seems to be what im doing. Maybe that is good enough.  

In [ ]:
def get_medoids(distance_matrix, labels):
    medoids = {}
    for label in set(labels):
        if label == -1:
            continue
        mask = np.where(labels == label)[0]
        sub_matrix = distance_matrix[np.ix_(mask, mask)]
        medoid_local_idx = sub_matrix.sum(axis=1).argmin()
        medoids[label] = mask[medoid_local_idx]
    return medoids  # original indices

In [ ]:
medoids = get_medoids(distance_matrix, hdbscan.labels_)

In [ ]:
l2comp = {label: all_comps[comp_idx] for label, comp_idx in medoids.items()}

In [ ]:
comps = []
titles = []
for label, comp_idx in medoids.items():
    comp = all_comps[comp_idx]
    comps.append(comp.reshape(SHAPE))
    comp_id = idx_by_comp_id[comp_idx]
    score, active_ratio = comp2score[comp_id]
    titles.append(f"L({label}) ({score:.3f}/{active_ratio:.3f})")

In [ ]:
# checked, these are perfect.
S(comps, (20, 15), 6, ax_titles=titles, mode=MODE)
plt.show()

In [ ]:
scaled_samples.shape, np.array(comps).shape

In [ ]:
torch.mps.empty_cache()
gc.collect()

In [ ]:
# now we train a model with these medoids as initialsations for the decoder.  
# everything else goes normally
# we specifically init the encoder with sigma_enc separately.
# this requires some work which ive done before, but oh well.
from pt_to_api.benchmark import train_x as TX
from pt_to_api import benchmark as B

comp_arr = np.array([c.reshape(-1) for c in comps]).copy()
model = TX.Autoencoder(scaled_samples.shape[1], comp_arr.shape[0])
model.decoder.weight.data = torch.tensor(comp_arr.T)

run = TX.train(
    scaled_samples,
    comp_arr.shape[0],
    1e-2,
    device="mps",
    epochs=1000,
    baseline_epochs=600,
    batch_size=64,
    use_ln_term=False,
    recon_err_schedule=TX.CosineAnnealReconError(1000),
    initialised_model=model,
    init_strategy=B.OnlyInitEncoderStrategy(),
)

In [ ]:
torch.save(run, "./run-random-weights-algo.pt")
torch.save(comp_arr, "./original-clustered-comps.pt")

In [ ]:
# if we look at the point of these operations, the main point below is to remove L(7) and L(8) competetion
# and realising the L(6) is shit
# Why is L(6) bad? it already does not explain much, so descent would replace it with something better by decomposing
# Why do L(7) and L(8) change? Cuz they are overlapping (technically, they are just the same things)

# currently, Im running the model again to do this. but is that wise?

# i keep coming back to the fact that i can simply select the min loss thing at some components. check out many seeds which are close at loss
# and keep the components which are stable. and train a new model with those initialised.
# my current problem is me going through clustering multiple n-comps, which might not be the best solution
# i guess we back to simply using good seeds
# now the problem right now is i need more seeds, im not getting stable losses.  
# so the next step does seem like ill need to work on JAX

In [ ]:
# original, show again
S(comps, (20, 9), 6, ax_titles=titles)
plt.show()

In [ ]:
# the main problem is splitting. how do we encourage no splitting?
# i do see what all is going away, but the decomposition is quite annoying.
# there should be no decomposition, only going to 0 or not
# what if we train only the encoder? 
# the rationale being that the components which are bigger forms of some other component would have overlaps.
# and as such, wont be used?
S([c.reshape(SHAPE) for c in run.components], (20, 9), 6, ax_titles=[])
plt.show()

In [ ]:
run.components.shape, np.array(comps).shape

In [ ]:
np.concat([run.components, np.array([c.reshape(-1) for c in comps])])

In [ ]:
distance_matrix = get_distance_matrix(all_comps)
hdbscan = HDBSCAN(copy=True, min_cluster_size=5, metric="precomputed")
hdbscan.fit(distance_matrix)